# Deduplication & Reference Date

This notebook handles two important data prep steps:

1. **Deduplication:** Keep only the oldest renewal record per customer (Co_Ref)
2. **Reference Date:** Calculate the reference date for each customer

The reference date is important because it's the point in time we use to find the nearest call or email interaction. It's either the actual close date or the renewal date + 28 days for customers who never closed.

## Setup

Bringing in our config, helpers, and loading the raw data.

In [0]:
%run ../02_config/config_setup
%run ../utils/data_helpers
%run ./load_tables

## Deduplication Strategy

Some customers appear multiple times in the billings table (multiple renewal cycles). For modeling, we want to focus on their **first renewal**, so we keep the record with the earliest Prospect_Renewal_Date.

This ensures we're predicting first-time churn behavior, not influenced by prior renewal experiences.

In [0]:
section("Deduplication: Keep oldest Co_Ref record")

# Record counts before deduplication
before = len(billings)

# Sort by renewal date and keep the first (oldest) record per customer
billings = (billings
            .sort_values('Prospect_Renewal_Date', ascending=True)
            .drop_duplicates(subset='Co_Ref', keep='first'))

after = len(billings)
removed = before - after

print(f"  Before : {before:,} rows")
print(f"  After  : {after:,} unique Co_Ref")
print(f"  Removed: {removed:,} duplicate rows (kept oldest renewal per customer)")
print(f"\n  Deduplication rate: {pct(removed, before):.1f}% of records were duplicates")

## Calculate Reference Date

The reference date is the point in time we use to find the nearest interaction (call or email). It's calculated as:

- **If customer closed:** Use the actual Closed_Date
- **If customer never closed:** Use Prospect_Renewal_Date + 28 days

Why 28 days? That's our churn threshold. A customer who hasn't closed within 28 days of their renewal date is considered churned.

In [0]:
section("Computing reference_date per customer")

# First calculate days between renewal date and close date
billings['days_to_close'] = (
    billings['Closed_Date'] - billings['Prospect_Renewal_Date']
).dt.days

# Reference date logic:
# - Use Closed_Date if available
# - Otherwise use Prospect_Renewal_Date + CHURN_DAYS_THRESHOLD
billings['reference_date'] = billings['Closed_Date'].fillna(
    billings['Prospect_Renewal_Date'] + pd.Timedelta(days=CHURN_DAYS_THRESHOLD)
)

# Count how many used each method
using_closed = billings['Closed_Date'].notna().sum()
using_prd_plus = billings['Closed_Date'].isna().sum()

print(f"  Reference date source breakdown:")
print(f"    Using Closed_Date          : {using_closed:,} customers ({pct(using_closed, len(billings)):.1f}%)")
print(f"    Using PRD + {CHURN_DAYS_THRESHOLD} days (no close): {using_prd_plus:,} customers ({pct(using_prd_plus, len(billings)):.1f}%)")
print(f"    Null reference_date        : {billings['reference_date'].isna().sum()}")

## Summary

We now have a clean, deduplicated billings table with reference dates calculated.

In [0]:
print("\n" + "="*65)
print("  DEDUPLICATION COMPLETE")
print("="*65)
print(f"\n  billings DataFrame: {billings.shape[0]:,} rows × {billings.shape[1]} cols")
print(f"  - Unique customers (Co_Ref): {billings['Co_Ref'].nunique():,}")
print(f"  - Reference date calculated for all records")
print(f"\n  Next step: Run churn_labeling to apply the 3-rule definition")